# Tensor-native N4 bias field correction

Correct the real `r16` image used in the bias-correction section of the [ANTsX tutorial](https://gist.github.com/ntustison/12a656a5fc2f6f9c4494c88dc09c5621).
The sample is downloaded on first use and resized to 128 × 128 for a short
example. No synthetic anatomy or simulated bias field is added.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

In [ ]:
from antstorch.bspline_flows import ImageDomain, n4_bias_field_correction_tensor

reference = ants.image_read(ants.get_ants_data("r16")).clone("float")
# Keep the real anatomy; reduce resolution only to shorten this tutorial.
reference = ants.resample_image(reference, (128, 128), use_voxels=True)
reference_mask = ants.get_mask(reference)
image = torch.from_numpy(reference.numpy().T.copy())[None, None].to(device)
mask = torch.from_numpy(reference_mask.numpy().T.copy())[None, None].to(device)
domain = ImageDomain(reference.shape, spacing=reference.spacing, origin=reference.origin,
                     direction=tuple(map(tuple, reference.direction)))
options = dict(shrink_factor=2, convergence={"iters": [10, 10], "tol": 1e-7},
               spline_param=(4, 4), number_of_histogram_bins=64)
corrected = n4_bias_field_correction_tensor(image, domain, mask=mask, **options)
bias = n4_bias_field_correction_tensor(image, domain, mask=mask,
                                      return_bias_field=True, **options)
assert corrected.shape == image.shape and torch.isfinite(corrected).all()
assert torch.isfinite(bias).all() and (bias > 0).all()

## Inspect correction and the estimated bias field
The input has real anatomical intensity variation: a lower global coefficient
of variation is not an accuracy criterion. Inspect tissue contrast, the mask and
the estimated smooth field. No ground-truth bias field is available for r16.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
for ax, values, title in zip(axes, [image, corrected, bias],
                             ["Tutorial r16", "N4 corrected", "Estimated bias"]):
    view = ax.imshow(values[0, 0].detach().cpu(), origin="lower", cmap="gray")
    ax.set_title(title)
    fig.colorbar(view, ax=ax, shrink=0.7)
plt.show()

This short demonstration schedule is not a convergence prescription for every
MRI. Choose the mask, resolution, spline spacing and convergence schedule for
the intended data. The image and mask use ANTs `(x, y)` geometry, while tensors
use `(batch, channels, y, x)`.